## Setup

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.status()

In [ ]:
using CairoMakie
using Carlo.ResultTools
using CarloAnalysis
using DataFrames
using FFTW
using HDF5
using JLD2
using LinearAlgebra
using StaticArrays

In [ ]:
function generate_spins(jobname, task_no)
    fig = Figure(size=(400, 400))

    task_str = lpad(task_no, 4, "0")
    h5open("../eta-jobs/$jobname.data/task$task_str/run0001.dump.h5") do file
        spins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        spin_xs = map(v -> v[1], spins)
        spin_ys = map(v -> v[2], spins)
        spin_zs = map(v -> v[3], spins)
        Lx, Ly = size(spins)
        fig[1,1] = Axis(fig; title="Spins", backgroundcolor="black")
        strength = vec(spin_zs)
        arrows2d!(1:Lx, 1:Ly, spin_xs, spin_ys, lengthscale=0.5, align=:center, color=strength)
    end

    return fig
end

In [ ]:
function generate_spinks(jobname, task_no; run_no=1, dimer=1)
    fig = Figure(size=(400, 400))

    task_str = lpad(task_no, 4, "0")
    run_str = lpad(run_no, 4, "0")
    h5open("../dimer-jobs/$jobname.data/task$task_str/run$run_str.dump.h5") do file
        rawetas = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        etas = zeros(size(rawetas)..., 3)
        for I in eachindex(IndexCartesian(), rawetas)
            etas[I, :] = rawetas[I]
        end
        etaks = fft(etas, (1, 2)) / length(rawetas)
        spin_mags = sum(abs2.(etaks), dims=3)[:,:,1]
        fig[1,1] = ax = Axis(fig; title="ηk correlations")
        heatmap!(ax, spin_mags, colorrange=(0,1))
    end

    return fig
end

## VBS Init

### Normal Runs

In [ ]:
results = JobResult("../dimer-jobs", "vbs")

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Dimer M'(blue) and 3K/4(orange)", xlabel="T")
generate_plot!(ax, :T, :sk_corr_part_K, results.data) do sk
    sk[3]
end
generate_plot!(ax, :T, :sk_corr_M2, results.data) do sk
    sk[3]
end
fig[1,2] = ax = Axis(fig, title="In plane η M'(blue) and 3K/4(orange)", xlabel="T")
generate_plot!(ax, :T, :ηk_corr_M2, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
generate_plot!(ax, :T, :ηk_corr_part_K, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
fig

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Dimer fraction", xlabel="T")
generate_plot!(ax, :T, :sk_Γ, results.data) do sk
    2 * real(sum(sk))
end
fig[1,2] = ax = Axis(fig, title="In plane η Γ expectation", xlabel="T", ylabel=L"\langle\eta^z\rangle")
generate_plot!(ax, :T, :ηk_corr_Γ, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
fig

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Energy", xlabel="T", ylabel="Energy")
generate_plot!(ax, :T, :Energy, results.data)
fig[1,2] = ax = Axis(fig, title="Heat Capacity", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax, :T, :HeatCap, results.data)
fig

In [ ]:
generate_spinks("vbs", 10, dimer=3)

In [ ]:
mctimes = get_mctime_data(results, :ηxy, :ηk_corr_M2, :ηk_corr_part_K, :sk_corr_part_K, :sk_corr_M2, :sk_Γ, :Energy)
nothing

In [ ]:
i = 1
fig = Figure(size=(800, 400))
data = mctimes[i]
fig[1,1] = ax = Axis(fig, title="η∥ M' and 3K/4 correlations", xlabel="Bin #", ylabel=L"\langle\eta_\parallel\rangle")
lines!(ax, real.(getindex.(data[:, :ηk_corr_M2], 1, 1) .+ getindex.(data[:, :ηk_corr_M2], 2, 2)))
lines!(ax, real.(getindex.(data[:, :ηk_corr_part_K], 1, 1) .+ getindex.(data[:, :ηk_corr_part_K], 2, 2)))
fig[1,2] = ax = Axis(fig, title="Dimer M' and 3K/4 correlations", xlabel="Bin #")
lines!(ax, getindex.(data[:, :sk_corr_M2], 3))
lines!(ax, getindex.(data[:, :sk_corr_part_K], 3))
fig

In [ ]:
i = 5
fig = Figure(size=(800, 400))
data = mctimes[i]
fig[1,1] = ax = Axis(fig, title="η∥ expectation", xlabel="Bin #", ylabel=L"\langle\eta_\parallel\rangle")
lines!(ax, data[:, :ηxy])
fig[1,2] = ax = Axis(fig, title="Dimer count", xlabel="Bin #")
lines!(ax, real.(sum.(data[:, :sk_Γ])))
fig

In [ ]:
i = 4
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Energy", xlabel="Bin #", ylabel="Energy")
lines!(ax, mctimes[i][:, :Energy])
fig[1,2] = ax = Axis(fig, title="Energy", xlabel="Bin #")
lines!(ax, mctimes[i+1][:, :Energy])
fig

### Parallel Tempered Runs

In [ ]:
results = JobResult("../dimer-jobs", "vbs-parallel")
df = results.data
results

In [ ]:
Ts = df[1, :parallel_tempering]["values"]

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Dimer M'(blue) and 3K/4(orange)", xlabel="T")
generate_plot!(ax, Ts, eachslice(df[1, :sk_corr_part_K], dims=2)) do sk
    sk[3]
end
generate_plot!(ax, Ts, eachslice(df[1, :sk_corr_M2], dims=2)) do sk
    sk[3]
end
fig[1,2] = ax = Axis(fig, title="In plane η M'(blue) and 3K/4(orange)", xlabel="T")
generate_plot!(ax, Ts, eachslice(df[1, :ηk_corr_M2], dims=3)) do ηk
    real(ηk[1,1] + ηk[2,2])
end
generate_plot!(ax, Ts, eachslice(df[1, :ηk_corr_part_K], dims=3)) do ηk
    real(ηk[1,1] + ηk[2,2])
end
fig

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Dimer fraction", xlabel="T")
generate_plot!(ax, Ts, eachslice(df[1, :sk_Γ], dims=2)) do sk
    2 * real(sum(sk))
end
fig[1,2] = ax = Axis(fig, title="In plane η Γ expectation", xlabel="T", ylabel=L"\langle\eta^z\rangle")
generate_plot!(ax, Ts, eachslice(df[1, :ηk_corr_Γ], dims=3)) do ηk
    real(ηk[1,1] + ηk[2,2])
end
fig

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Energy", xlabel="T", ylabel="Energy")
generate_plot!(ax, Ts, df[1, :Energy])
fig[1,2] = ax = Axis(fig, title="Heat Capacity", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax, Ts, df[1, :HeatCap])
fig

In [ ]:
generate_spinks("vbs", 10, dimer=3)

In [ ]:
mctimes = get_mctime_data(results, :ηk_corr_Γ, :ηk_corr_M2, :ηk_corr_part_K, :sk_corr_part_K, :sk_corr_M2, :sk_Γ, :Energy)
nothing

In [ ]:
i = 8
fig = Figure(size=(800, 400))
data = mctimes[1]

fig[1,1] = ax = Axis(fig, title="η∥ Γ correlations", xlabel="Bin #")
lines!(ax, real.(getindex.(data[:, :ηk_corr_Γ], 1, 1, i) .+ getindex.(data[:, :ηk_corr_M2], 2, 2, i)))
fig[1,2] = ax = Axis(fig, title="Dimer M' and 3K/4 correlations", xlabel="Bin #")
lines!(ax, real.(sum.(getindex.(data[:, :sk_Γ], :, i))))

fig

In [ ]:
i = 4
fig = Figure(size=(800, 400))
data = mctimes[1]

fig[1,1] = ax = Axis(fig, title="Energy", xlabel="Bin #", ylabel="Energy")
lines!(ax, getindex.(data[:, :Energy], i))
fig[1,2] = ax = Axis(fig, title="Energy", xlabel="Bin #")
lines!(ax, getindex.(data[:, :Energy], i+1))

fig

## Rand Init

In [ ]:
results = JobResult("../dimer-jobs", "vbs-anneal")

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Dimer M'(blue) and 3K/4(orange)", xlabel="T")
generate_plot!(ax, :T, :sk_corr_part_K, results.data) do sk
    sk[3]
end
generate_plot!(ax, :T, :sk_corr_M2, results.data) do sk
    sk[3]
end
fig[1,2] = ax = Axis(fig, title="In plane η M'(blue) and 3K/4(orange)", xlabel="T")
generate_plot!(ax, :T, :ηk_corr_M2, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
generate_plot!(ax, :T, :ηk_corr_part_K, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
fig

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Dimer count", xlabel="T")
generate_plot!(ax, :T, :sk_Γ, results.data) do sk
    real(sum(sk))
end
fig[1,2] = ax = Axis(fig, title="In plane η Γ expectation", xlabel="T")
generate_plot!(ax, :T, :ηk_corr_Γ, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
fig

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Energy", xlabel="T", ylabel="Energy")
generate_plot!(ax, :T, :Energy, results.data)
fig[1,2] = ax = Axis(fig, title="Heat Capacity", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax, :T, :HeatCap, results.data)
fig

In [ ]:
generate_spinks("vbs-anneal", 1, dimer=3)

## Eta-Only

In [ ]:
results = JobResult("../dimer-jobs", "vbs-eta")

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="In plane η M'(blue) and 3K/4(orange)", xlabel="T")
generate_plot!(ax, :T, :ηk_corr_M2, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
generate_plot!(ax, :T, :ηk_corr_part_K, results.data) do ηk
    real(ηk[1,1] + ηk[2,2])
end
fig[1,2] = ax = Axis(fig, title="In plane RMS η Γ correlation", xlabel="T")
generate_plot!(ax, :T, :ηk_corr_Γ, results.data) do ηk
    sqrt(real(ηk[1,1] + ηk[2,2]))
end
save("plots/vbs_eta_corrs.png", fig)
fig

In [ ]:
fig = Figure(size=(800, 400))
fig[1,1] = ax = Axis(fig, title="Energy", xlabel="T", ylabel="Energy")
generate_plot!(ax, :T, :Energy, results.data)
fig[1,2] = ax = Axis(fig, title="Heat Capacity", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax, :T, :HeatCap, results.data)
save("plots/vbs_eta_energy.png", fig)
fig

In [ ]:
generate_spinks("vbs", 1, dimer=2)

In [ ]:
mctimes = get_mctime_data(results, :ηxy, :ηk_corr_M2, :Energy)
nothing

In [ ]:
i = 1
fig = Figure(size=(800, 400))
data = mctimes[i]
fig[1,1] = ax = Axis(fig, title="In plane η expectation", xlabel="Bin #", ylabel=L"\langle\eta_\parallel\rangle")
lines!(ax, data[:, :ηxy])
fig[1,2] = ax = Axis(fig, title="Sk correlation M2", xlabel="Bin #")
lines!(ax, data[:, :Energy])
fig